# Model Training and Evaluation

## Objective

Train and evaluate a machine learning model to classify network connections as normal or malicious using the NSL-KDD dataset.

## Dataset

We will use the cleaned NSL-KDD training and testing datasets prepared during the preprocessing stage. The training dataset will be divided into training and validation subsets to support model development and comparison.

The official test dataset will remain separate and will be used for the final evaluation after selecting the model configuration.

## Planned workflow

1. Load the cleaned training and testing datasets.
2. Split the training data into training and validation subsets.
3. Prepare the features and targets, fitting preprocessing components on the training subset only.
4. Verify the data dimensions and class distributions.
5. Train a baseline Random Forest model.
6. Evaluate the baseline using validation data.
7. Analyze missed attacks and investigate unseen attack categories.
8. Compare model improvements using validation results.
9. Evaluate the selected model on the official test dataset.
10. Save the selected model and its preprocessing components for dashboard integration.

## Important note

The validation dataset will guide model development and model selection. The official test dataset will not be used to train models or choose between experiments. It will be reserved for final evaluation.

## Step 1: Load the Original NSL-KDD Datasets

### Objective

Load the original NSL-KDD training and testing files before splitting or encoding the data.

### Explanation

Each record contains 41 network traffic features, an original attack label, and a difficulty score, giving us 43 columns.

We will load both files without assuming they contain column headers. We will then assign the appropriate column names.

The original training dataset will later be split into training and validation subsets. The official test dataset will remain separate for final evaluation.

At this stage, we are only loading the data; we are not training a model or fitting an encoder.

In [1]:

import pandas as pd
from pathlib import Path

# Locate the processed dataset directory.
processed_dir = Path("../datasets/processed")

# Load the cleaned datasets.
train_clean = pd.read_csv(processed_dir / "train_clean.csv")
test_clean = pd.read_csv(processed_dir / "test_clean.csv")

# Inspect dataset dimensions.
print("Training dataset shape:", train_clean.shape)
print("Testing dataset shape:", test_clean.shape)

# Inspect available columns.
print("\nTraining columns:")
print(train_clean.columns.tolist())

print("\nTesting columns:")
print(test_clean.columns.tolist())

# Display the first five training records.
display(train_clean.head())


Training dataset shape: (125973, 44)
Testing dataset shape: (22544, 44)

Training columns:
['duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations', 'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login', 'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate', 'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'label', 'difficulty', 'target']

Testing columns:
['duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,difficulty,target
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20,0
1,0,udp,other,SF,146,0,0,0,0,0,...,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15,0
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19,1
3,0,tcp,http,SF,232,8153,0,0,0,0,...,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21,0
4,0,tcp,http,SF,199,420,0,0,0,0,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21,0


## Step 2: Split the Training Dataset into Training and Validation Sets

### Objective

Divide the cleaned training dataset into two subsets: one for training the machine learning model and another for validating its performance during development.

### Why this step is necessary

The training subset teaches the model to distinguish normal network connections from attacks. The validation subset helps us measure how well the model performs on records it did not use during training.

We will allocate 80% of the original training records to training and 20% to validation. We will use stratified splitting to preserve approximately the same proportion of normal and attack records in both subsets.

### Important considerations

- We will split the original training dataset before fitting any encoder or training any model.
- The `target` column will be used for stratification so both subsets maintain a representative class distribution.
- The official test dataset will remain separate and will not be used for model selection.
- We will retain the original attack labels for later analysis, but they will not be used as model input features.

In [2]:

from sklearn.model_selection import train_test_split

# Split the cleaned training records into training and validation sets.
train_data, validation_data = train_test_split(
    train_clean,
    test_size=0.20,
    random_state=42,
    stratify=train_clean["target"]
)

# Reset row indexes after splitting.
train_data = train_data.reset_index(drop=True)
validation_data = validation_data.reset_index(drop=True)

# Display the dimensions of each subset.
print("Training subset shape:", train_data.shape)
print("Validation subset shape:", validation_data.shape)
print("Official test dataset shape:", test_clean.shape)


Training subset shape: (100778, 44)
Validation subset shape: (25195, 44)
Official test dataset shape: (22544, 44)


## Step 3: Separate Features, Targets, and Original Labels

### Objective

Separate the network traffic features from the target variable in the training, validation, and official test datasets.

### Explanation

Machine learning models need input features (`X`) and the correct answers (`y`) during training.

Our dataset contains three columns that require special handling:

- **`target`:** The binary classification target. A value of `0` represents normal traffic, and `1` represents an attack.
- **`label`:** The original attack category, such as `normal` or `neptune`. We will preserve this separately for later analysis of missed attacks.
- **`difficulty`:** A dataset-specific difficulty score. We will exclude it from the model inputs because it is metadata rather than one of the 41 network traffic features.

We will remove `target`, `label`, and `difficulty` from the input features. This leaves the 41 network traffic features expected by the NSL-KDD dataset.

We will perform the same separation for the training subset, validation subset, and official test dataset so their structures remain consistent.

In [3]:

# Separate the binary targets.
y_train = train_data["target"]
y_validation = validation_data["target"]
y_test = test_clean["target"]

# Preserve the original attack categories for later analysis.
labels_train = train_data["label"]
labels_validation = validation_data["label"]
labels_test = test_clean["label"]

# Remove target and metadata columns from the model inputs.
columns_to_exclude = ["target", "label", "difficulty"]

X_train_raw = train_data.drop(columns=columns_to_exclude)
X_validation_raw = validation_data.drop(columns=columns_to_exclude)
X_test_raw = test_clean.drop(columns=columns_to_exclude)

# Display the resulting dimensions.
print("Training features:", X_train_raw.shape)
print("Validation features:", X_validation_raw.shape)
print("Test features:", X_test_raw.shape)

print("\nTraining targets:", y_train.shape)
print("Validation targets:", y_validation.shape)
print("Test targets:", y_test.shape)


Training features: (100778, 41)
Validation features: (25195, 41)
Test features: (22544, 41)

Training targets: (100778,)
Validation targets: (25195,)
Test targets: (22544,)


## Step 4: Verify the Target Class Distributions

### Objective

Inspect the distribution of normal and attack records in the training, validation, and official test datasets.

### Explanation

Our binary target uses:

- `0` to represent normal network traffic.
- `1` to represent malicious network traffic.

Stratified splitting was used to preserve the class proportions when creating the training and validation subsets. We will now verify this by counting the records in each class.

This check is important for intrusion detection because a model can appear to perform well if one class dominates the dataset. We need to understand how many normal and malicious records are available before training.

The official test set comes from the separate NSL-KDD test dataset, so its class distribution may differ from the training and validation subsets.

In [4]:

# Count normal and attack records in each subset.
print("Training target distribution:")
print(y_train.value_counts().sort_index().rename({
    0: "Normal",
    1: "Attack"
}))

print("\nValidation target distribution:")
print(y_validation.value_counts().sort_index().rename({
    0: "Normal",
    1: "Attack"
}))

print("\nOfficial test target distribution:")
print(y_test.value_counts().sort_index().rename({
    0: "Normal",
    1: "Attack"
}))

# Display class proportions as percentages.
print("\nClass proportions (%):")

for name, targets in [
    ("Training", y_train),
    ("Validation", y_validation),
    ("Test", y_test)
]:
    proportions = (
        targets.value_counts(normalize=True)
        .sort_index()
        .mul(100)
        .rename(index={0: "Normal", 1: "Attack"})
        .round(2)
    )

    print(f"\n{name}:")
    print(proportions)


Training target distribution:
target
Normal    53874
Attack    46904
Name: count, dtype: int64

Validation target distribution:
target
Normal    13469
Attack    11726
Name: count, dtype: int64

Official test target distribution:
target
Normal     9711
Attack    12833
Name: count, dtype: int64

Class proportions (%):

Training:
target
Normal    53.46
Attack    46.54
Name: proportion, dtype: float64

Validation:
target
Normal    53.46
Attack    46.54
Name: proportion, dtype: float64

Test:
target
Normal    43.08
Attack    56.92
Name: proportion, dtype: float64


## Step 5: Encode the Categorical Features

### Objective

Convert categorical network features into a numeric representation that the Random Forest classifier can use.

### Explanation

The NSL-KDD dataset contains three categorical input features:

- `protocol_type`
- `service`
- `flag`

One-hot encoding creates a separate binary column for each category. For example, a protocol value such as `tcp` will be represented by a numeric indicator column rather than text.

### Avoiding data leakage

We will fit the encoder using only the training subset. We will then use that same fitted encoder to transform the validation and official test subsets.

The encoder will use `handle_unknown="ignore"` so that a category appearing in validation or test data but not in the training subset will not cause an error. This is especially important because NSL-KDD contains attack and service categories that may differ across subsets.

We will keep the encoded data in sparse matrix format to use memory efficiently.

In [5]:

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from scipy import sparse

# Identify categorical input features.
categorical_features = ["protocol_type", "service", "flag"]

# Identify numerical input features.
numerical_features = [
    column for column in X_train_raw.columns
    if column not in categorical_features
]

print("Categorical features:", categorical_features)
print("Number of numerical features:", len(numerical_features))

# Create a preprocessing pipeline.
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        ("numerical", "passthrough", numerical_features)
    ],
    remainder="drop"
)

# Fit the encoder on training data only.
X_train = preprocessor.fit_transform(X_train_raw)

# Transform validation and test data using the fitted encoder.
X_validation = preprocessor.transform(X_validation_raw)
X_test = preprocessor.transform(X_test_raw)

# Convert the outputs to sparse matrices if needed.
X_train = sparse.csr_matrix(X_train)
X_validation = sparse.csr_matrix(X_validation)
X_test = sparse.csr_matrix(X_test)

# Display encoded feature dimensions.
print("\nEncoded training features:", X_train.shape)
print("Encoded validation features:", X_validation.shape)
print("Encoded test features:", X_test.shape)


Categorical features: ['protocol_type', 'service', 'flag']
Number of numerical features: 38

Encoded training features: (100778, 121)
Encoded validation features: (25195, 121)
Encoded test features: (22544, 121)


## Step 6: Validate the Prepared Features and Targets

### Objective

Confirm that the encoded feature matrices have the expected dimensions and that their corresponding target labels contain valid binary values.

### Explanation

Before training a model, we must ensure that:

- The number of feature rows matches the number of target labels in each dataset.
- All three encoded matrices have the same number of columns.
- The target labels contain only `0` and `1`.
- The preprocessing step produced numeric features suitable for model training.

These checks help identify data preparation errors before they affect model training or evaluation.

In [7]:
import numpy as np

# Check that feature rows match target counts.
assert X_train.shape[0] == len(y_train)
assert X_validation.shape[0] == len(y_validation)
assert X_test.shape[0] == len(y_test)

# Check that all feature matrices have the same number of columns.
assert X_train.shape[1] == X_validation.shape[1] == X_test.shape[1]

# Check that the target labels are binary.
assert set(y_train.unique()) == {0, 1}
assert set(y_validation.unique()) == {0, 1}
assert set(y_test.unique()) == {0, 1}

# Check that stored feature values are finite.
assert np.isfinite(X_train.data).all()
assert np.isfinite(X_validation.data).all()
assert np.isfinite(X_test.data).all()

print("All validation checks passed!")
print("Training feature count:", X_train.shape[1])
print("Target classes:", sorted(y_train.unique()))

All validation checks passed!
Training feature count: 121
Target classes: [np.int64(0), np.int64(1)]


### 8. Train the Baseline Random Forest Model

**Purpose:** Train our first machine-learning model to distinguish normal network traffic from malicious traffic.

**How it works:**
- A Random Forest combines predictions from multiple decision trees.
- Each tree learns patterns from the training data.
- The model combines the trees' predictions to classify network connections.
- We train using `X_train` and `y_train` only. The validation and official test datasets remain separate.

**Why start with a baseline?** This gives us a reference point. Later, we can compare it with an improved model and determine whether our changes actually help.

**Expected result:** A confirmation message when training finishes. Training time will depend on your computer's hardware.

In [8]:
from sklearn.ensemble import RandomForestClassifier

# Create the baseline model.
baseline_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# Train using only the training subset.
baseline_model.fit(X_train, y_train)

print("Baseline Random Forest training completed!")

Baseline Random Forest training completed!


### 9. Evaluate the Baseline Model on the Validation Dataset

**Purpose:** Measure how well our trained model distinguishes normal network connections from attacks on data it did not train on.

**Metrics we will examine:**
- **Accuracy:** The proportion of all predictions that are correct.
- **Precision:** Of the connections predicted to be attacks, how many actually are attacks?
- **Recall:** Of all actual attacks, how many did the model detect?
- **F1-score:** A balance between precision and recall.
- **Confusion matrix:** Shows correct and incorrect predictions for each class.

We will use the validation dataset for this evaluation. We will not use the official test dataset to choose or tune our model.

For this project, pay particular attention to **attack recall** because missed attacks are important in an intrusion detection system.

In [9]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Predict labels for the validation dataset.
y_validation_pred = baseline_model.predict(X_validation)

# Calculate overall accuracy.
baseline_accuracy = accuracy_score(y_validation, y_validation_pred)

print("Baseline Random Forest — Validation Results")
print("=" * 45)
print(f"Accuracy: {baseline_accuracy:.4f}")
print("\nClassification Report:")
print(
    classification_report(
        y_validation,
        y_validation_pred,
        labels=[0, 1],
        target_names=["Normal", "Attack"],
        digits=4
    )
)

print("Confusion Matrix:")
print(confusion_matrix(y_validation, y_validation_pred, labels=[0, 1]))

Baseline Random Forest — Validation Results
Accuracy: 0.9990

Classification Report:
              precision    recall  f1-score   support

      Normal     0.9987    0.9996    0.9991     13469
      Attack     0.9995    0.9985    0.9990     11726

    accuracy                         0.9990     25195
   macro avg     0.9991    0.9990    0.9990     25195
weighted avg     0.9990    0.9990    0.9990     25195

Confusion Matrix:
[[13463     6]
 [   18 11708]]


### 10. Check for Duplicate Records Across Training and Validation

**Purpose:** Determine whether identical records occur in both subsets, which could make our validation results overly optimistic.

**How it works:** We calculate a hash for each complete row in the training and validation datasets, then count validation rows whose matching hash also exists in training.

**Important:** This is a diagnostic check, not proof of leakage by itself. If duplicates exist, we will investigate how they affect our evaluation before drawing conclusions about model performance.

In [11]:
# Create hashes for complete rows in both subsets.
train_hashes = set(
    pd.util.hash_pandas_object(train_data, index=False)
)

validation_hashes = pd.util.hash_pandas_object(
    validation_data, index=False
)

# Count validation rows whose complete records also occur in training.
overlap_mask = validation_hashes.isin(train_hashes)
overlapping_rows = int(overlap_mask.sum())

print("Validation rows:", len(validation_data))
print("Rows also found in training:", overlapping_rows)
print(
    "Percentage of validation rows overlapping:",
    f"{overlapping_rows / len(validation_data) * 100:.2f}%"
)

Validation rows: 25195
Rows also found in training: 0
Percentage of validation rows overlapping: 0.00%


### 11. Check for Identical Network Features Across Splits

**Purpose:** Check whether identical network feature combinations occur in both the training and validation subsets.

**How it works:** We calculate row hashes using only the 41 network features. Labels and metadata are excluded so they cannot hide matching feature vectors.

**Interpretation:** If there is no overlap, that's reassuring. If there is overlap, it does not automatically invalidate the model, but we should investigate before making strong claims about generalization.

In [12]:
# Hash only the original network features.
train_feature_hashes = set(
    pd.util.hash_pandas_object(X_train_raw, index=False)
)

validation_feature_hashes = pd.util.hash_pandas_object(
    X_validation_raw, index=False
)

# Count validation feature vectors also present in training.
feature_overlap_mask = validation_feature_hashes.isin(
    train_feature_hashes
)

feature_overlap_count = int(feature_overlap_mask.sum())

print("Validation rows:", len(X_validation_raw))
print("Rows with features also found in training:", feature_overlap_count)
print(
    "Feature overlap percentage:",
    f"{feature_overlap_count / len(X_validation_raw) * 100:.2f}%"
)

Validation rows: 25195
Rows with features also found in training: 5
Feature overlap percentage: 0.02%


### 12. Analyze Attack Categories in the Validation Dataset

**Purpose:** Understand which attack types appear in the validation data and how many examples of each type are present.

**How it works:** We use the original `label` column, which we preserved before encoding, to count normal traffic and individual attack categories.

**Why it matters:** A model can perform well at binary attack detection without necessarily performing equally well on every attack type. This analysis helps us understand the coverage of our evaluation dataset.

We will analyze the validation data only at this stage; the official test set remains untouched.

In [13]:
# Count original labels in the training and validation subsets.
print("Attack categories in training data:")
print(labels_train.value_counts().to_string())

print("\nAttack categories in validation data:")
print(labels_validation.value_counts().to_string())

print("\nNumber of distinct labels:")
print("Training:", labels_train.nunique())
print("Validation:", labels_validation.nunique())

Attack categories in training data:
label
normal             53874
neptune            33068
satan               2871
ipsweep             2864
portsweep           2337
smurf               2128
nmap                1195
back                 760
teardrop             693
warezclient          692
pod                  160
guess_passwd          40
buffer_overflow       25
warezmaster           15
land                  13
rootkit               10
imap                   9
loadmodule             8
ftp_write              6
perl                   3
multihop               3
phf                    3
spy                    1

Attack categories in validation data:
label
normal             13469
neptune             8146
satan                762
ipsweep              735
portsweep            594
smurf                518
nmap                 298
teardrop             199
warezclient          198
back                 196
pod                   41
guess_passwd          13
buffer_overflow        5
warezmaster  

### 13. Evaluate Detection Recall by Attack Category

**Purpose:** Determine the proportion of validation examples detected as attacks for each original attack label.

**How it works:** We compare the model's binary predictions with the original labels. For each attack type, we calculate the percentage classified as `Attack`.

**Important:** This measures binary attack detection for each category, not whether the model identifies the specific attack type. A prediction of `Attack` is considered correct for this analysis whenever the original label is an attack.

In [14]:
# Combine original validation labels with the model's predictions.
validation_results = pd.DataFrame({
    "original_label": labels_validation.reset_index(drop=True),
    "actual_target": y_validation.reset_index(drop=True),
    "predicted_target": y_validation_pred
})

# Calculate detection recall for each attack category.
attack_recall_by_category = (
    validation_results[validation_results["actual_target"] == 1]
    .groupby("original_label")
    .agg(
        total_examples=("actual_target", "size"),
        detected_attacks=("predicted_target", "sum")
    )
)

attack_recall_by_category["detection_recall_%"] = (
    attack_recall_by_category["detected_attacks"]
    / attack_recall_by_category["total_examples"]
    * 100
).round(2)

attack_recall_by_category = attack_recall_by_category.sort_values(
    "detection_recall_%"
)

display(attack_recall_by_category)

,total_examples,detected_attacks,detection_recall_%
original_label,,,
ftp_write,2,0,0.00
spy,1,0,0.00
multihop,4,2,50.00
guess_passwd,13,12,92.31
warezclient,198,193,97.47
pod,41,40,97.56
portsweep,594,589,99.16
satan,762,761,99.87
loadmodule,1,1,100.00


### 14. Identify Underrepresented Attack Categories

**Purpose:** Identify attack categories with fewer than 20 examples in the validation set.

**How it works:** We use the original validation labels to count examples per attack category and filter for categories with small sample sizes.

**Why it matters:** Results for rare attacks can be unstable. This check helps us distinguish genuine performance concerns from results based on very few examples.

This is an analysis step only; it does not modify our datasets or model.

In [16]:
# Count validation examples for each original attack category.
validation_attack_counts = (
    validation_results[validation_results["actual_target"] == 1]
    ["original_label"]
    .value_counts()
    .sort_values()
)

# Show categories with fewer than 20 validation examples.
rare_validation_attacks = validation_attack_counts[
    validation_attack_counts < 20
]

print("Attack categories with fewer than 20 validation examples:")
display(rare_validation_attacks.to_frame(name="validation_examples"))

print("\nNumber of underrepresented categories:", len(rare_validation_attacks))

Attack categories with fewer than 20 validation examples:


,validation_examples
original_label,
loadmodule,1
spy,1
phf,1
ftp_write,2
imap,2
multihop,4
land,5
buffer_overflow,5
warezmaster,5



Number of underrepresented categories: 10


### 15. Train a Class-Weighted Random Forest

**Purpose:** Test whether class weighting improves binary attack detection compared with the baseline.

**How it works:** The `balanced` setting automatically adjusts class weights inversely to their frequencies in the training labels. The model then trains on the same training subset as the baseline.

**Fair comparison:** We will evaluate this model on the same validation dataset and compare its accuracy, attack precision, attack recall, and F1-score with the baseline.

**Important:** This does not specifically rebalance rare attack categories, because all attack types share the binary label `Attack`.

In [17]:
from sklearn.ensemble import RandomForestClassifier

# Create a class-weighted Random Forest.
weighted_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

# Train using the same training subset.
weighted_model.fit(X_train, y_train)

print("Class-Weighted Random Forest training completed!")

Class-Weighted Random Forest training completed!


### 16. Evaluate the Class-Weighted Random Forest

**Purpose:** Measure whether class weighting improves the model's ability to distinguish normal traffic from attacks.

**How it works:** We generate predictions for the validation dataset and calculate accuracy, precision, recall, F1-score, and the confusion matrix.

**Comparison:** We will compare these results with the baseline model's validation metrics. We will not use the official test set to select the model.

**Key metric:** Attack recall is especially important because it measures the proportion of actual attacks that our model detects.

In [20]:
# Generate validation predictions.
y_weighted_pred = weighted_model.predict(X_validation)

# Calculate the weighted model's metrics.
weighted_accuracy = accuracy_score(y_validation, y_weighted_pred)

print("Class-Weighted Random Forest — Validation Results")
print("=" * 50)
print(f"Accuracy: {weighted_accuracy:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_validation,
        y_weighted_pred,
        labels=[0, 1],
        target_names=["Normal", "Attack"],
        digits=4
    )
)

print("Confusion Matrix:")
print(confusion_matrix(y_validation, y_weighted_pred, labels=[0, 1]))

# Compare the two models using their validation predictions.
baseline_report = classification_report(
    y_validation, y_validation_pred,
    labels=[0, 1], output_dict=True
)
weighted_report = classification_report(
    y_validation, y_weighted_pred,
    labels=[0, 1], output_dict=True
)

comparison = pd.DataFrame({
    "Baseline": {
        "Accuracy": accuracy_score(y_validation, y_validation_pred),
        "Attack Precision": baseline_report["1"]["precision"],
        "Attack Recall": baseline_report["1"]["recall"],
        "Attack F1-score": baseline_report["1"]["f1-score"]
    },
    "Class-Weighted": {
        "Accuracy": weighted_accuracy,
        "Attack Precision": weighted_report["1"]["precision"],
        "Attack Recall": weighted_report["1"]["recall"],
        "Attack F1-score": weighted_report["1"]["f1-score"]
    }
})

print("\nModel Comparison:")
display(comparison.round(4))

Class-Weighted Random Forest — Validation Results
Accuracy: 0.9992

Classification Report:
              precision    recall  f1-score   support

      Normal     0.9989    0.9996    0.9992     13469
      Attack     0.9995    0.9987    0.9991     11726

    accuracy                         0.9992     25195
   macro avg     0.9992    0.9991    0.9992     25195
weighted avg     0.9992    0.9992    0.9992     25195

Confusion Matrix:
[[13463     6]
 [   15 11711]]

Model Comparison:


,Baseline,Class-Weighted
Accuracy,0.9990,0.9992
Attack Precision,0.9995,0.9995
Attack Recall,0.9985,0.9987
Attack F1-score,0.9990,0.9991


### 17. Summarize the Validation Model Comparison

**Purpose:** Create a concise comparison of both trained models using the same validation dataset.

**How it works:** We collect accuracy, attack precision, attack recall, attack F1-score, and the number of missed attacks into one table.

**Decision:** We will use these validation results to nominate a provisional model. The final assessment will come after evaluating the selected model on the official NSL-KDD test dataset.

In [21]:
# Build a comparison table from the existing predictions.
model_comparison = pd.DataFrame({
    "Baseline": {
        "Accuracy": accuracy_score(y_validation, y_validation_pred),
        "Attack Precision": baseline_report["1"]["precision"],
        "Attack Recall": baseline_report["1"]["recall"],
        "Attack F1-score": baseline_report["1"]["f1-score"],
        "Missed Attacks": int(
            ((y_validation == 1) & (y_validation_pred == 0)).sum()
        )
    },
    "Class-Weighted": {
        "Accuracy": accuracy_score(y_validation, y_weighted_pred),
        "Attack Precision": weighted_report["1"]["precision"],
        "Attack Recall": weighted_report["1"]["recall"],
        "Attack F1-score": weighted_report["1"]["f1-score"],
        "Missed Attacks": int(
            ((y_validation == 1) & (y_weighted_pred == 0)).sum()
        )
    }
})

display(model_comparison.round(4))

,Baseline,Class-Weighted
Accuracy,0.9990,0.9992
Attack Precision,0.9995,0.9995
Attack Recall,0.9985,0.9987
Attack F1-score,0.9990,0.9991
Missed Attacks,18.0000,15.0000


### 18. Evaluate the Selected Model on the Official Test Dataset

**Purpose:** Measure how well the class-weighted Random Forest generalizes to the official NSL-KDD test dataset.

**How it works:** We use the pre-fitted preprocessing pipeline to transform the test features, then generate predictions with the selected model. We calculate accuracy, precision, recall, F1-score, and a confusion matrix.

**Important:** The official test set has not been used to train either model or select between them. This evaluation gives us a more independent assessment of performance on the supplied test dataset.

After this evaluation, we should avoid repeatedly tuning the model based on test results. If further changes are needed, we should return to the training and validation process.

In [22]:
# Generate predictions on the official test dataset.
y_test_pred = weighted_model.predict(X_test)

# Calculate official test accuracy.
test_accuracy = accuracy_score(y_test, y_test_pred)

print("Class-Weighted Random Forest — Official Test Results")
print("=" * 55)
print(f"Accuracy: {test_accuracy:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_test_pred,
        labels=[0, 1],
        target_names=["Normal", "Attack"],
        digits=4
    )
)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred, labels=[0, 1]))

Class-Weighted Random Forest — Official Test Results
Accuracy: 0.7709

Classification Report:
              precision    recall  f1-score   support

      Normal     0.6585    0.9728    0.7854      9711
      Attack     0.9678    0.6182    0.7544     12833

    accuracy                         0.7709     22544
   macro avg     0.8131    0.7955    0.7699     22544
weighted avg     0.8345    0.7709    0.7678     22544

Confusion Matrix:
[[9447  264]
 [4900 7933]]


### 19. Compare Detection of Seen and Unseen Attack Categories

**Purpose:** Investigate whether the model's test performance differs between attack categories represented in training and categories absent from training.

**How it works:** We compare the original attack labels in the training and official test datasets, then calculate binary attack recall separately for the two groups.

**Interpretation:** Poorer recall on unseen categories would suggest a generalization challenge. However, good recall on both groups would not rule out other distribution differences.

**Important:** This is diagnostic analysis of the official test results we have already obtained. We will not use these results as a repeatedly tuned validation metric.

In [23]:
# Identify attack categories represented in training.
training_attack_categories = set(labels_train.unique()) - {"normal"}

# Identify attack categories present in the official test set.
test_attack_categories = set(labels_test.unique()) - {"normal"}

# Find categories absent from the training subset.
unseen_attack_categories = test_attack_categories - training_attack_categories

print("Attack categories in training:", len(training_attack_categories))
print("Attack categories in official test:", len(test_attack_categories))
print("Test attack categories absent from training:", len(unseen_attack_categories))
print("\nUnseen attack categories:")
print(sorted(unseen_attack_categories))

# Evaluate attack recall for categories seen and unseen during training.
test_attack_results = pd.DataFrame({
    "original_label": labels_test.reset_index(drop=True),
    "actual_target": y_test.reset_index(drop=True),
    "predicted_target": y_test_pred
})

attack_rows = test_attack_results[test_attack_results["actual_target"] == 1].copy()

attack_rows["category_group"] = attack_rows["original_label"].apply(
    lambda label: (
        "Seen in training"
        if label in training_attack_categories
        else "Unseen in training"
    )
)

group_results = attack_rows.groupby("category_group").agg(
    total_attacks=("actual_target", "size"),
    detected_attacks=("predicted_target", "sum")
)

group_results["attack_recall_%"] = (
    group_results["detected_attacks"]
    / group_results["total_attacks"]
    * 100
).round(2)

display(group_results)

Attack categories in training: 22
Attack categories in official test: 37
Test attack categories absent from training: 17

Unseen attack categories:
['apache2', 'httptunnel', 'mailbomb', 'mscan', 'named', 'processtable', 'ps', 'saint', 'sendmail', 'snmpgetattack', 'snmpguess', 'sqlattack', 'udpstorm', 'worm', 'xlock', 'xsnoop', 'xterm']


,total_attacks,detected_attacks,attack_recall_%
category_group,,,
Seen in training,9083,6955,76.57
Unseen in training,3750,978,26.08


### 20. Identify Attack Categories With the Most Missed Detections

**Purpose:** Find the attack categories contributing most to false negatives in the official test evaluation.

**How it works:** We group actual test attacks by their original category, count how many were detected and missed, and calculate recall for each category.

**Why it matters:** This helps us distinguish categories that are difficult to detect from categories with high miss counts simply because they appear frequently.

**Evaluation note:** These results are diagnostic findings from the official test set. We will use them to describe limitations, not to repeatedly tune the model against the test set.

In [24]:
# Count test attacks, detections, and missed attacks by original category.
test_attack_category_results = (
    test_attack_results[test_attack_results["actual_target"] == 1]
    .groupby("original_label")
    .agg(
        total_attacks=("actual_target", "size"),
        detected_attacks=("predicted_target", "sum")
    )
)

test_attack_category_results["missed_attacks"] = (
    test_attack_category_results["total_attacks"]
    - test_attack_category_results["detected_attacks"]
)

test_attack_category_results["recall_%"] = (
    test_attack_category_results["detected_attacks"]
    / test_attack_category_results["total_attacks"]
    * 100
).round(2)

# Show the categories responsible for the largest numbers of missed attacks.
test_attack_category_results = test_attack_category_results.sort_values(
    "missed_attacks", ascending=False
)

display(test_attack_category_results.head(15))

,total_attacks,detected_attacks,missed_attacks,recall_%
original_label,,,,
guess_passwd,1231,0,1231,0.00
warezmaster,944,117,827,12.39
processtable,685,12,673,1.75
apache2,737,177,560,24.02
mscan,996,448,548,44.98
snmpguess,331,0,331,0.00
mailbomb,293,0,293,0.00
snmpgetattack,178,0,178,0.00
httptunnel,133,17,116,12.78


### 21. Baseline Findings and Limitations

**Validation performance**
- Baseline Random Forest accuracy: 99.90%.
- Class-weighted Random Forest accuracy: 99.92%.
- The class-weighted model missed 15 validation attacks, compared with 18 for the baseline.

**Official test performance**
- Accuracy: 77.09%.
- Attack precision: 96.78%.
- Attack recall: 61.82%.
- Attack F1-score: 75.44%.
- Missed attacks: 4,900 out of 12,833.

**Key findings**
1. The official test dataset contains 17 attack categories absent from the training subset.
2. Attack recall was 76.57% for categories represented in training and 26.08% for categories absent from training.
3. Several categories had very low or zero detection recall, including `guess_passwd`, `snmpguess`, and `mailbomb`.
4. The difference between validation and test performance indicates a generalization challenge that requires further investigation.

**Limitations**
- The model performs binary classification and does not identify individual attack families.
- Some rare attack categories have very few training examples.
- NSL-KDD is a benchmark dataset; these results do not establish equivalent performance on live network traffic.

**Next steps**
- Investigate and compare alternative models using training and validation data.
- Evaluate candidate models with attack recall, precision, F1-score, and confusion matrices.
- Select a candidate before performing a final official test evaluation.
- Integrate the selected model into the Streamlit dashboard after saving the model and preprocessing pipeline.

### 22. Plan the Model Improvement Experiment

**Objective:** Improve the model's ability to detect malicious network connections, particularly attacks that are difficult to recognize.

**Experimental rules**
1. Use the existing training subset to fit candidate models.
2. Use the validation subset to compare candidate models.
3. Keep the official test set out of further model selection.
4. Compare accuracy, attack precision, attack recall, attack F1-score, and the confusion matrix.
5. Record each experiment so that the final model choice is evidence-based.

**First experiment:** Train a Decision Tree classifier as an alternative to Random Forest. A Decision Tree is simpler to interpret and gives us a useful comparison against our existing models.

**Important:** We will not assume that a different algorithm will perform better. We will measure its performance on the validation set.

In [25]:
from sklearn.tree import DecisionTreeClassifier

# Create a Decision Tree with balanced class weights.
decision_tree_model = DecisionTreeClassifier(
    random_state=42,
    class_weight="balanced"
)

# Train using the training subset only.
decision_tree_model.fit(X_train, y_train)

print("Decision Tree training completed!")

Decision Tree training completed!


### 23. Evaluate the Decision Tree on the Validation Dataset

**Purpose:** Measure how well the Decision Tree detects normal connections and attacks on data it did not train on.

**How it works:** We generate predictions for the validation subset and calculate accuracy, precision, recall, F1-score, and the confusion matrix.

**Comparison:** We will compare the Decision Tree with the baseline and class-weighted Random Forest models using the same validation examples.

**Key consideration:** Attack recall is particularly important because it measures the proportion of actual attacks correctly detected.

In [26]:
# Generate predictions on the validation dataset.
y_tree_pred = decision_tree_model.predict(X_validation)

# Calculate and display Decision Tree performance.
tree_accuracy = accuracy_score(y_validation, y_tree_pred)

print("Decision Tree — Validation Results")
print("=" * 40)
print(f"Accuracy: {tree_accuracy:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_validation,
        y_tree_pred,
        labels=[0, 1],
        target_names=["Normal", "Attack"],
        digits=4
    )
)

print("Confusion Matrix:")
print(confusion_matrix(y_validation, y_tree_pred, labels=[0, 1]))

# Compare all three models on the same validation subset.
tree_report = classification_report(
    y_validation,
    y_tree_pred,
    labels=[0, 1],
    output_dict=True
)

model_comparison = pd.DataFrame({
    "Baseline RF": {
        "Accuracy": accuracy_score(y_validation, y_validation_pred),
        "Attack Precision": baseline_report["1"]["precision"],
        "Attack Recall": baseline_report["1"]["recall"],
        "Attack F1-score": baseline_report["1"]["f1-score"]
    },
    "Weighted RF": {
        "Accuracy": accuracy_score(y_validation, y_weighted_pred),
        "Attack Precision": weighted_report["1"]["precision"],
        "Attack Recall": weighted_report["1"]["recall"],
        "Attack F1-score": weighted_report["1"]["f1-score"]
    },
    "Decision Tree": {
        "Accuracy": tree_accuracy,
        "Attack Precision": tree_report["1"]["precision"],
        "Attack Recall": tree_report["1"]["recall"],
        "Attack F1-score": tree_report["1"]["f1-score"]
    }
})

print("\nValidation Model Comparison:")
display(model_comparison.round(4))

Decision Tree — Validation Results
Accuracy: 0.9987

Classification Report:
              precision    recall  f1-score   support

      Normal     0.9990    0.9984    0.9987     13469
      Attack     0.9982    0.9989    0.9986     11726

    accuracy                         0.9987     25195
   macro avg     0.9986    0.9987    0.9986     25195
weighted avg     0.9987    0.9987    0.9987     25195

Confusion Matrix:
[[13448    21]
 [   13 11713]]

Validation Model Comparison:


,Baseline RF,Weighted RF,Decision Tree
Accuracy,0.9990,0.9992,0.9987
Attack Precision,0.9995,0.9995,0.9982
Attack Recall,0.9985,0.9987,0.9989
Attack F1-score,0.9990,0.9991,0.9986


### 25. Train an Extra Trees Classifier

**Purpose:** Test another tree-ensemble algorithm to see whether it generalizes better than our existing models.

**How it works:** Extra Trees trains many randomized decision trees and combines their predictions. Its tree-splitting strategy differs from Random Forest, so it may learn different decision boundaries.

**Experiment rules:** We will train only on `X_train` and `y_train`, then evaluate on the same validation subset.

**Important:** A stronger validation score does not guarantee better performance on unseen attack types. The official test set will remain reserved for the final evaluation of our chosen candidate.

In [27]:
from sklearn.ensemble import ExtraTreesClassifier

# Create the Extra Trees model.
extra_trees_model = ExtraTreesClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

# Train using the training subset only.
extra_trees_model.fit(X_train, y_train)

print("Extra Trees training completed!")

Extra Trees training completed!


### 26. Evaluate Extra Trees on the Validation Dataset

**Purpose:** Determine whether Extra Trees improves binary attack detection compared with our existing candidates.

**How it works:** We generate predictions on the validation subset and calculate accuracy, precision, recall, F1-score, and a confusion matrix.

**Fair comparison:** We use the same validation examples and metrics as the previous experiments. The official test set remains reserved for final evaluation.

**Key consideration:** We will examine attack recall alongside precision and F1-score rather than relying on accuracy alone.

In [28]:
# Generate predictions on the validation dataset.
y_extra_trees_pred = extra_trees_model.predict(X_validation)

# Calculate and display Extra Trees performance.
extra_trees_accuracy = accuracy_score(y_validation, y_extra_trees_pred)

print("Extra Trees — Validation Results")
print("=" * 40)
print(f"Accuracy: {extra_trees_accuracy:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_validation,
        y_extra_trees_pred,
        labels=[0, 1],
        target_names=["Normal", "Attack"],
        digits=4
    )
)

print("Confusion Matrix:")
print(confusion_matrix(y_validation, y_extra_trees_pred, labels=[0, 1]))

# Extract metrics for comparison.
extra_trees_report = classification_report(
    y_validation,
    y_extra_trees_pred,
    labels=[0, 1],
    output_dict=True
)

# Compare all four models on the same validation subset.
model_comparison = pd.DataFrame({
    "Baseline RF": {
        "Accuracy": accuracy_score(y_validation, y_validation_pred),
        "Attack Precision": baseline_report["1"]["precision"],
        "Attack Recall": baseline_report["1"]["recall"],
        "Attack F1-score": baseline_report["1"]["f1-score"]
    },
    "Weighted RF": {
        "Accuracy": accuracy_score(y_validation, y_weighted_pred),
        "Attack Precision": weighted_report["1"]["precision"],
        "Attack Recall": weighted_report["1"]["recall"],
        "Attack F1-score": weighted_report["1"]["f1-score"]
    },
    "Decision Tree": {
        "Accuracy": accuracy_score(y_validation, y_tree_pred),
        "Attack Precision": tree_report["1"]["precision"],
        "Attack Recall": tree_report["1"]["recall"],
        "Attack F1-score": tree_report["1"]["f1-score"]
    },
    "Extra Trees": {
        "Accuracy": extra_trees_accuracy,
        "Attack Precision": extra_trees_report["1"]["precision"],
        "Attack Recall": extra_trees_report["1"]["recall"],
        "Attack F1-score": extra_trees_report["1"]["f1-score"]
    }
})

print("\nValidation Model Comparison:")
display(model_comparison.round(4))

Extra Trees — Validation Results
Accuracy: 0.9989

Classification Report:
              precision    recall  f1-score   support

      Normal     0.9987    0.9992    0.9990     13469
      Attack     0.9991    0.9986    0.9988     11726

    accuracy                         0.9989     25195
   macro avg     0.9989    0.9989    0.9989     25195
weighted avg     0.9989    0.9989    0.9989     25195

Confusion Matrix:
[[13458    11]
 [   17 11709]]

Validation Model Comparison:


,Baseline RF,Weighted RF,Decision Tree,Extra Trees
Accuracy,0.9990,0.9992,0.9987,0.9989
Attack Precision,0.9995,0.9995,0.9982,0.9991
Attack Recall,0.9985,0.9987,0.9989,0.9986
Attack F1-score,0.9990,0.9991,0.9986,0.9988


### 27. Select the Provisional Model for Integration

**Selected candidate:** Class-Weighted Random Forest.

**Reason for selection:** It achieved the highest validation accuracy (99.92%) and attack F1-score (99.91%) among the four evaluated candidates.

**Important limitation:** On the official NSL-KDD test set, this model achieved 77.09% accuracy and 61.82% attack recall. Its performance on unseen attack categories was particularly weak.

**Deployment status:** This is a provisional research model, not a validated production-grade intrusion detector. The Streamlit dashboard should display these limitations and must not present demo data as real network detections.

**Next step:** Save the fitted model and its preprocessing pipeline so the dashboard can load them consistently.

In [29]:
# Select the provisional model for integration.
selected_model = weighted_model
selected_model_name = "Class-Weighted Random Forest"

print("Selected model:", selected_model_name)
print("Validation accuracy:", f"{accuracy_score(y_validation, selected_model.predict(X_validation)):.4f}")
print("Official test accuracy:", f"{test_accuracy:.4f}")
print("Official test attack recall:", f"{weighted_report['1']['recall']:.4f}")

Selected model: Class-Weighted Random Forest
Validation accuracy: 0.9992
Official test accuracy: 0.7709
Official test attack recall: 0.9987


### 27. Confirm the Selected Model and Its Evaluation Results

**Purpose:** Verify the selected model's validation and official test performance using the correct predictions and labels.

**How it works:** We calculate validation metrics from `y_validation` and `y_weighted_pred`, and official test metrics from `y_test` and `y_test_pred`. These must remain separate.

**Expected results:** Validation accuracy of approximately 99.92%, official test accuracy of 77.09%, and official test attack recall of 61.82%.

In [30]:
# Recalculate metrics from the correct datasets.
official_test_report = classification_report(
    y_test,
    y_test_pred,
    labels=[0, 1],
    target_names=["Normal", "Attack"],
    output_dict=True,
    zero_division=0
)

print("Selected model:", selected_model_name)
print(
    "Validation accuracy:",
    f"{accuracy_score(y_validation, y_weighted_pred):.4f}"
)
print(
    "Official test accuracy:",
    f"{accuracy_score(y_test, y_test_pred):.4f}"
)
print(
    "Official test attack recall:",
    f"{official_test_report['Attack']['recall']:.4f}"
)

Selected model: Class-Weighted Random Forest
Validation accuracy: 0.9992
Official test accuracy: 0.7709
Official test attack recall: 0.6182


### 28. Save the Selected Model and Preprocessing Pipeline

**Purpose:** Save the fitted class-weighted Random Forest and its preprocessing pipeline so they can be reused without retraining.

**What we will save:**
- `intrusion_detection_model.joblib` — the trained classifier.
- `preprocessor.joblib` — the fitted `ColumnTransformer` and `OneHotEncoder`.

**Why both files matter:** The dashboard must convert incoming network features using exactly the same encoding learned during training. Using a different encoder or fitting a new one could produce incorrect predictions.

**Destination:** The project's `models/` directory.

**Important:** Only load these files from trusted sources. Joblib files use Python serialization and can execute code when loaded.

In [31]:
import joblib
from pathlib import Path

# Locate the project root and create the models directory.
project_root = Path.cwd().parent
models_dir = project_root / "models"
models_dir.mkdir(parents=True, exist_ok=True)

# Save the selected trained model and fitted preprocessor.
model_path = models_dir / "intrusion_detection_model.joblib"
preprocessor_path = models_dir / "preprocessor.joblib"

joblib.dump(selected_model, model_path)
joblib.dump(preprocessor, preprocessor_path)

print("Model saved to:", model_path.resolve())
print("Preprocessor saved to:", preprocessor_path.resolve())
print("\nSaved artifacts:")
print("Model exists:", model_path.exists())
print("Preprocessor exists:", preprocessor_path.exists())

Model saved to: C:\Users\NIPSEY\Desktop\AI-Intrusion-Detection-System\models\intrusion_detection_model.joblib
Preprocessor saved to: C:\Users\NIPSEY\Desktop\AI-Intrusion-Detection-System\models\preprocessor.joblib

Saved artifacts:
Model exists: True
Preprocessor exists: True


### 29. Verify the Saved Model and Preprocessor

**Purpose:** Confirm that the saved artifacts can be loaded and used to make predictions.

**How it works:** We load both files from the `models/` directory and compare predictions from the loaded model against predictions from the original model on a small sample of validation data.

**Expected result:** Both files load successfully, and predictions from the loaded model match the original model.

This is a verification check, not a new model evaluation. We will not retrain the model or change the official test results.

In [32]:
import joblib

# Load the saved artifacts.
loaded_model = joblib.load(model_path)
loaded_preprocessor = joblib.load(preprocessor_path)

# Compare predictions on a small validation sample.
sample_features = X_validation[:10]

original_predictions = selected_model.predict(sample_features)
loaded_predictions = loaded_model.predict(sample_features)

assert (original_predictions == loaded_predictions).all()

print("Model loaded successfully!")
print("Preprocessor loaded successfully!")
print("Prediction consistency check passed!")
print("Sample predictions:", loaded_predictions.tolist())

Model loaded successfully!
Preprocessor loaded successfully!
Prediction consistency check passed!
Sample predictions: [1, 0, 1, 1, 0, 0, 0, 0, 1, 1]


### 30. Prepare the Streamlit Dashboard Integration

**Purpose:** Identify how the existing dashboard loads its pages, displays metrics, and presents detection alerts before connecting the trained model.

**Plan:**
1. Inspect the current dashboard code.
2. Confirm the location of the saved model and preprocessor.
3. Add model-loading logic.
4. Connect predictions to the dashboard.
5. Test the integration using sample data before considering live network traffic.

**Important:** The model currently expects the 41 NSL-KDD network features in the same format used during training. We must not send arbitrary data or the dashboard's demo metrics directly to the model and call the results real detections.

### 31. Prepare the Streamlit Model Integration

**Purpose:** Establish reliable paths to the trained model and preprocessing artifacts.

**Plan:** Resolve artifact paths relative to the dashboard script, verify both files exist, and load them without retraining.

**Safety:** Keep the existing demo data clearly labelled. Predictions will only be generated from inputs containing the expected NSL-KDD features; we will not treat demo records as actual detections.